# Feature Engineering
Builds and selects user, tweet, and graph features for the bot detection model.
Split into three sections:

Section 1 — Core Account + Tweet Behavior Features
- account age
- follower/friend ratio
- profile completeness
- tweet count
- retweet ratio
- hashtag/mention/URL averages
- text length averages
- save base feature dataset

Section 2 — Graph/Network Features
- load follower/friend/edge files
- standardize edges
- compute in-degree/out-degree
- compute graph degree
- compute reciprocity
- optional PageRank/community features
- merge with Round 1 dataset
- save graph-enhanced dataset

Section 3 — AI/Text Authenticity Features
- analyze tweet text for AI-like or synthetic-writing patterns
- compute per-account text authenticity features
- merge with Round 2 dataset
- save final model-ready dataset

These answer the three questions:
- Section 1: What does the account look like?
- Section 2: How is the account connected?
- Section 3: How natural or synthetic does the account’s language look?

In [10]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..")
sys.path.append(str(PROJECT_ROOT))

from utils.base_features import (
    ROUND1_FEATURE_COLS,
    available_feature_cols,
    build_round1_features_for_dataset,
    print_round1_summary,
    save_feature_frame,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

PREPROCESSED_DIR = PROJECT_ROOT / "data" / "pre-processed"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

ROUND1_DIR = PROCESSED_DIR / "round1_base"
ROUND1_DIR.mkdir(parents=True, exist_ok=True)

## Section 1: Base Features

In [11]:
DATASETS = {
    "cresci_2015": {
        "users": PREPROCESSED_DIR / "cresci_2015" / "users_cresci_2015.csv",
        "tweets": PREPROCESSED_DIR / "cresci_2015" / "tweets_cresci_2015.csv",
        "reference_date": "2015-12-31",
    },
    "cresci_2017": {
        "users": PREPROCESSED_DIR / "cresci_2017" / "users_cresci_2017.csv",
        "tweets": PREPROCESSED_DIR / "cresci_2017" / "tweets_cresci_2017.csv",
        "reference_date": "2017-12-31",
    },
    "twibot_2020": {
        "users": PREPROCESSED_DIR / "twibot_2020" / "users_twibot_20.csv",
        "tweets": PREPROCESSED_DIR / "twibot_2020" / "tweets_twibot_20.csv",
        "reference_date": "2020-12-31",
    },
    "twibot_2022": {
        "users": PREPROCESSED_DIR / "twibot_2022" / "users_twibot_22.csv",
        "tweets": PREPROCESSED_DIR / "twibot_2022" / "tweets_twibot_22.csv",
        "reference_date": "2022-12-31",
    },
}

for name, paths in DATASETS.items():
    print(name)
    print("  users:", paths["users"].exists(), paths["users"])
    print("  tweets:", paths["tweets"].exists(), paths["tweets"])

cresci_2015
  users: True ../data/pre-processed/cresci_2015/users_cresci_2015.csv
  tweets: True ../data/pre-processed/cresci_2015/tweets_cresci_2015.csv
cresci_2017
  users: True ../data/pre-processed/cresci_2017/users_cresci_2017.csv
  tweets: True ../data/pre-processed/cresci_2017/tweets_cresci_2017.csv
twibot_2020
  users: True ../data/pre-processed/twibot_2020/users_twibot_20.csv
  tweets: True ../data/pre-processed/twibot_2020/tweets_twibot_20.csv
twibot_2022
  users: True ../data/pre-processed/twibot_2022/users_twibot_22.csv
  tweets: True ../data/pre-processed/twibot_2022/tweets_twibot_22.csv


### Build and save Round 1 features

In [12]:
round1_frames = []

for dataset_name, paths in DATASETS.items():
    print(f"\nBuilding Round 1 features for {dataset_name}")

    round1 = build_round1_features_for_dataset(dataset_name, paths)

    out_path = ROUND1_DIR / f"{dataset_name}_base_features.csv"
    save_feature_frame(round1, out_path)

    print(f"  Shape: {round1.shape}")
    print(f"  Saved: {out_path}")

    round1_frames.append(round1)


Building Round 1 features for cresci_2015


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/notebooks/../utils/base_features.py:58: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)


  Shape: (5301, 23)
  Saved: ../data/processed/round1_base/cresci_2015_base_features.csv

Building Round 1 features for cresci_2017


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/notebooks/../utils/base_features.py:58: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)


  Shape: (14368, 23)
  Saved: ../data/processed/round1_base/cresci_2017_base_features.csv

Building Round 1 features for twibot_2020


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/notebooks/../utils/base_features.py:58: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)


  Shape: (11826, 23)
  Saved: ../data/processed/round1_base/twibot_2020_base_features.csv

Building Round 1 features for twibot_2022
  Shape: (1000000, 23)
  Saved: ../data/processed/round1_base/twibot_2022_base_features.csv


In [13]:
round1_all = pd.concat(round1_frames, ignore_index=True, sort=False)

combined_path = ROUND1_DIR / "all_datasets_base_features.csv"
save_feature_frame(round1_all, combined_path)

print(f"Combined Round 1 shape: {round1_all.shape}")
print(f"Saved combined Round 1 file to: {combined_path}")

Combined Round 1 shape: (1031495, 23)
Saved combined Round 1 file to: ../data/processed/round1_base/all_datasets_base_features.csv


### Sanity check

In [14]:
print_round1_summary(round1_all)

round1_feature_cols = available_feature_cols(round1_all)

print("\nFinal feature columns:")
for col in sorted(round1_all.columns):
    print("-", col)

print("\nMissing values in feature columns:")
display(
    round1_all[round1_feature_cols]
    .isna()
    .sum()
    .to_frame("missing_count")
)

Rows by dataset:
dataset
twibot_2022    1000000
cresci_2017      14368
twibot_2020      11826
cresci_2015       5301
Name: count, dtype: int64

Label distribution:
label           bot   human      All
dataset                             
cresci_2015    3987    1314     5301
cresci_2017   10894    3474    14368
twibot_2020    6589    5237    11826
twibot_2022  139943  860057  1000000
All          161413  870082  1031495

Round 1 feature columns:
- followers_count
- friends_count
- statuses_count
- favourites_count
- listed_count
- account_age_days
- ff_ratio
- has_bio
- has_location
- has_default_pic
- tweet_count_actual
- tweets_per_day
- avg_text_length
- retweet_ratio
- avg_hashtags
- avg_mentions
- avg_urls

Missing values in Round 1 feature columns:
followers_count             0
friends_count               0
statuses_count              0
favourites_count      1000000
listed_count                0
account_age_days         1000
ff_ratio                    0
has_bio                   

,missing_count
followers_count,0
friends_count,0
statuses_count,0
favourites_count,1000000
listed_count,0
account_age_days,1000
ff_ratio,0
has_bio,0
has_location,0
has_default_pic,0


### Preview Sample

In [16]:
pd.set_option("display.max_columns", None)

print("Random sample of Round 1 dataset:")
display(round1_all.sample(10, random_state=42))

print("\nDataset shape:", round1_all.shape)

Random sample of Round 1 dataset:


,user_id,label,subset,followers_count,friends_count,statuses_count,favourites_count,listed_count,verified,source,dataset,account_age_days,ff_ratio,has_bio,has_location,has_default_pic,tweet_count_actual,avg_text_length,retweet_ratio,avg_hashtags,avg_mentions,avg_urls,tweets_per_day
630470,u1406656820224180226,human,train,19,314,40,NaN,0,False,twibot_22,twibot_2022,558.0,0.060317,False,False,False,40.0,34.000000,0.000000,0.000000,1.150000,0.100000,0.071556
975609,u87559736,human,train,167,303,347,NaN,4,False,twibot_22,twibot_2022,4804.0,0.549342,True,True,False,41.0,132.121951,0.634146,0.146341,1.121951,0.219512,0.008533
357122,u1465943110656135168,human,val,5,9,0,NaN,0,False,twibot_22,twibot_2022,394.0,0.500000,True,False,False,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
410197,u27515885,human,train,368,445,15505,NaN,4,False,twibot_22,twibot_2022,5024.0,0.825112,True,False,False,196.0,74.204082,0.423469,0.102041,0.959184,0.423469,0.039005
784056,u3502528829,human,train,4075,1453,4754,NaN,64,True,twibot_22,twibot_2022,2678.0,2.802613,True,True,False,56.0,211.428571,0.125000,0.053571,0.660714,0.642857,0.020903
465305,u80415454,human,train,7305,7527,192552,NaN,210,False,twibot_22,twibot_2022,4833.0,0.970377,True,True,False,202.0,107.049505,0.311881,0.133663,0.603960,0.623762,0.041787
793531,u1046862909639409665,human,train,12803,1172,12560,NaN,24,False,twibot_22,twibot_2022,1551.0,10.914749,True,True,False,95.0,207.663158,0.378947,0.126316,2.978947,0.736842,0.061211
199444,u112019826,human,test,900,1322,3147,NaN,63,False,twibot_22,twibot_2022,4709.0,0.680272,True,True,False,61.0,88.000000,0.081967,0.262295,0.967213,0.508197,0.012951
793956,u1466609071298015236,bot,val,15,59,147,NaN,2,False,twibot_22,twibot_2022,392.0,0.250000,True,True,False,66.0,239.030303,0.045455,7.833333,0.348485,1.500000,0.167939
906519,u895726503660363776,human,val,431,731,3475,NaN,1,False,twibot_22,twibot_2022,1968.0,0.588798,True,True,False,42.0,173.047619,0.000000,3.261905,0.000000,2.000000,0.021331



Dataset shape: (1031495, 23)


## Section 2: Graph